# CEADS parser walkthrough

This notebook is the practical guide for parsing the currently supported CEADS China provincial MRIO workbooks in MARIO.

## What this notebook covers

- where the supported CEADS workbooks come from;
- which workbook family MARIO currently parses;
- what `format=` means;
- how to use `path=` and `year=`;
- how MARIO maps imports, exports, and the optional `CO2` row.

## Source pages

- CEADS portal: [Input-Output Tables](https://www.ceads.net/data/input_output_tables/)
- Scientific Data descriptor: [China's Provincial Multi-Regional Input-Output Database for 2018 and 2020](https://www.nature.com/articles/s41597-025-06543-y)
- Dataset DOI: [10.6084/m9.figshare.29927291](https://doi.org/10.6084/m9.figshare.29927291)

MARIO does not download the CEADS workbook automatically. Download the workbook locally before running this notebook.

## Main entry point

For normal user workflows, the public entry point is `mario.parse_ceads(...)`.

At the moment, that entry point supports only `table="IOT"`.

## What MARIO currently parses

The current backend is intentionally narrow:

- it parses local Excel workbooks only;
- it currently supports the verified CEADS provincial MRIO workbook family for `2018` and `2020`;
- it reads the English table sheet, not the Chinese one;
- it uses the `Sector` and `Province` sheets as metadata support.

## What `format` means

`format=` is the parser-side layout selector. It does not mean "which CEADS dataset" you are using; it means "which workbook structure" MARIO should expect.

At the moment, MARIO supports:

- `format="auto"`: inspect the workbook and resolve the known layout automatically;
- `format="ceads_provincial_workbook"`: force the verified 2018/2020 workbook structure.

## Configure local paths

Set `MARIO_DOC_DATA` to the root folder where you keep real parser datasets, or set `MARIO_CEADS_PATH` directly to the CEADS folder.

Expected local structure:

```text
CEADS/
├── MRIO 2018.xlsx
└── MRIO 2020.xlsx
```

In [ ]:
from pathlib import Path
import os

import mario

In [ ]:
DATA_ROOT = Path(os.environ.get("MARIO_DOC_DATA", ".")).expanduser()
CEADS_ROOT = Path(os.environ.get("MARIO_CEADS_PATH", DATA_ROOT / "CEADS")).expanduser()

CEADS_ROOT

## Parse one explicit workbook

Use this form when you want to parse a specific downloaded workbook.

In [ ]:
ceads_2018 = CEADS_ROOT / "MRIO 2018.xlsx"

db_2018 = mario.parse_ceads(
    path=ceads_2018,
    format="auto",
    calc_all=False,
)

db_2018

## Parse from a directory

Use this form when one folder contains multiple CEADS workbooks. In that case, `year=` selects which workbook to parse.

In [ ]:
db_2020 = mario.parse_ceads(
    path=CEADS_ROOT,
    year=2020,
    calc_all=False,
)

db_2020

## How the parsed table is mapped

For the verified CEADS workbook family:

- `Z` comes from the main intermediate-transactions block;
- `Y` contains the domestic final-demand block plus one `Exports` column per province;
- `V` contains `Imports` plus the four value-added rows exposed by the workbook;
- `E` contains the optional `CO2 (Mt)` row when it exists;
- `EY` is initialized at zero.

In [ ]:
sorted(db_2020["baseline"].keys())

In [ ]:
{matrix: block.shape for matrix, block in db_2020["baseline"].items()}

## inspect the parsed database

Once parsed, the result is a standard MARIO database.

In [ ]:
db_2020.get_index("Region")[:10]

In [ ]:
db_2020.get_index("Sector")[:10]

In [ ]:
db_2020.get_index("Satellite account")

## Caveats

- The CEADS portal lists older provincial MRIO releases too, such as `2012`, `2015`, and `2017`. MARIO does not assume those older workbook families are compatible until they are checked explicitly.
- The parser intentionally relies on the English worksheet. If only a Chinese worksheet is available, that workbook is not currently supported.
- The workbook metadata sheets can contain minor naming inconsistencies. For the verified 2018/2020 family, MARIO treats the main transaction axis as canonical when such conflicts appear.